# Drift and ablations

Point `EVAL_DIRS` at one `metrics.json` per run. The eval command already writes `RESULTS.md`. This notebook is for overlaying those curves. It does not invent numbers when the files are missing.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

paths = sorted(Path('outputs').glob('*/metrics.json'))
print(f'{len(paths)} metrics files')
for path in paths:
    summary = json.loads(path.read_text())
    drift = summary.get('drift') or {}
    if not drift:
        print(path, 'has no drift table')
        continue
    horizons = sorted(drift, key=int)
    metric = 'lpips' if 'lpips' in drift[horizons[0]] else 'mae'
    means = [drift[h][metric]['mean'] for h in horizons]
    plt.plot([int(h) for h in horizons], means, marker='o', label=path.parent.name)
plt.xlabel('horizon')
plt.ylabel(metric if paths else 'metric')
plt.legend()
plt.title('Free-running drift')
plt.show()
